# Notebook 1 · The RL loop and Gymnasium

**Time:** about an hour.

**You'll learn**
- what reinforcement learning is, and how it differs from the machine learning you may have met before
- which library does what
- how to drive an environment through Gymnasium's API
- how our balloon behaves when you drop ballast or vent helium

**How these notebooks work.** Run the cells from top to bottom with Shift+Enter. Cells marked ✏️ are
exercises: replace the `raise NotImplementedError` line with your own code. Most exercises are
followed by a check that prints ✅ when you've got it right. If you get stuck, every answer is in
`solutions/`. Sections marked 🚀 are optional stretch goals for anyone who finishes early.

> 📗 **This is the solutions copy.** The exercise version is `onboarding/01_rl_loop_and_gymnasium.ipynb`. Try the exercises there first.

## What is reinforcement learning?

In the machine learning you may have met already (*supervised* learning), you're given examples with
the right answers, such as photos labelled "cat" or "dog", and a model learns to reproduce the labels.

In **reinforcement learning (RL)** nobody gives you the right answers. An **agent** takes **actions**
in an **environment**, and the environment hands back a **reward**: a single number saying how well
things are going. The agent has to work out, by trial and error, which actions lead to the most reward
*over time*. That last part is what makes it hard: a good action might only pay off much later, and an
action that looks good now might cause trouble later.

```
        ┌──────────────── action ────────────────┐
        │                                        ▼
      agent                                environment
        ▲                                        │
        └──────── observation, reward ───────────┘
```

Every step, the agent sees an **observation**, picks an action, and gets back the next observation and
a reward. One run from start to finish is an **episode**.

| Term | Meaning | Balloon example |
|---|---|---|
| Observation | What the agent can see right now | altitude, vertical speed, ballast left |
| Action | What it can do | vent helium, do nothing, drop ballast |
| Reward | A number scoring the last step | +1 for every 3 minutes spent near the target altitude |
| Episode | One complete attempt | one 2-hour flight |
| Policy | The agent's rule for choosing actions | "drop ballast if too low and not rising" |
| Return | The total reward over an episode | what the agent is trying to maximise |

## Which library does what

| Library | What it's for | Where you'll use it |
|---|---|---|
| **Gymnasium** | The standard interface between agents and environments: `reset()` and `step()`. It contains environments, not learning algorithms. | Notebooks 1–4 |
| **NumPy, Matplotlib, pandas** | Arrays, plots and tables | Everywhere |
| **Stable-Baselines3 (SB3)** | Tested implementations of RL algorithms (DQN, PPO, …) that can train on any Gymnasium environment | Notebook 4 |
| **PyTorch** | The neural-network library SB3 is built on | Notebook 4 |
| **TensorBoard** | Live dashboards of training progress | Notebook 4 |
| **sb3-contrib** | Extra algorithms for SB3, including QR-DQN, which the main project uses | Mentioned in notebook 4 |

## Setup

Run the next two cells. The first lets the notebook find `toy_env.py`, a small helper file that sits
next to these notebooks. If the second fails with `No module named 'environments'`, the project isn't
installed in this Python environment: follow the install steps in the main README
(`pip install -e .[dev]`), then restart the kernel.

In [ ]:
import sys
from pathlib import Path

# toy_env.py lives in onboarding/. This makes it importable from both the
# exercise copy (onboarding/) and the solutions copy (onboarding/solutions/).
here = Path.cwd()
sys.path.insert(0, str(here.parent if here.name == "solutions" else here))

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from environments.core.balloon import Balloon
from toy_env import check

## Part 1 · Gymnasium with CartPole

Before the balloon, let's learn the API on CartPole, the "hello world" of RL. A pole is hinged to a
cart, and each step you push the cart left or right. You get +1 for every step the pole stays up. The
episode ends if the pole tilts more than 12° or the cart runs off the track, and is cut off after 500
steps.

In [ ]:
env = gym.make("CartPole-v1", render_mode="rgb_array")
print("Actions:     ", env.action_space)
print("Observations:", env.observation_space)

obs, info = env.reset(seed=0)
print("First observation:", obs)

plt.imshow(env.render())
plt.axis("off")
plt.show()

`Discrete(2)` means there are two actions, numbered 0 (push left) and 1 (push right). The `Box` with
shape `(4,)` means each observation is 4 numbers: cart position, cart velocity, pole angle (in radians)
and the pole's angular velocity.

`reset()` starts a new episode. It returns the first observation and an `info` dict of extra details
(empty for CartPole). Passing a `seed` makes the episode repeatable.

Now take one step:

In [ ]:
obs, reward, terminated, truncated, info = env.step(1)  # push right
print("observation:", obs)
print("reward:     ", reward)
print("terminated: ", terminated, "| truncated:", truncated)

`step(action)` returns five things:

- **observation:** what the agent sees next
- **reward:** the reward for that step
- **terminated:** the episode ended because of something that happened in it, like the pole falling over
- **truncated:** the episode was cut off from outside, usually by a time limit
- **info:** extra details, handy for debugging

The difference between `terminated` and `truncated` matters for learning. After a termination there is
no future reward to expect. After a truncation the world would have carried on; we just stopped
watching. Notebook 3 relies on this.

### ✏️ Exercise 1.1 · The agent–environment loop

Write `run_episode(env, policy, seed)`, which plays one full episode and returns the total reward. A
**policy** here is just a function that takes an observation and returns an action.

Reset the environment with the seed, then keep asking the policy for an action and stepping the
environment, adding up the rewards, until the episode is terminated or truncated.

In [ ]:
def run_episode(env, policy, seed):
    """Play one episode and return its total reward."""
    # BEGIN SOLUTION
    obs, info = env.reset(seed=seed)
    total_reward = 0.0
    done = False
    while not done:
        action = policy(obs)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        done = terminated or truncated
    return total_reward
    # END SOLUTION


def random_policy(obs):
    return env.action_space.sample()


def average_return(policy, n_episodes=20):
    return np.mean([run_episode(env, policy, seed) for seed in range(n_episodes)])

In [ ]:
random_score = average_return(random_policy)
print(f"Random policy: {random_score:.1f} steps on average")
check(10 <= random_score <= 40, "run_episode works: random play keeps the pole up for about 20 steps")

### ✏️ Exercise 1.2 · Write a policy by hand

Write a policy that keeps the pole up. The observation is
`[cart position, cart velocity, pole angle, pole angular velocity]`, and action `1` pushes the cart
right.

1. Start with `lean_policy`: push the cart in the direction the pole is leaning.
2. Then write `better_policy`, which also uses the pole's angular velocity. Try pushing according to
   the sign of `angle + 0.5 * angular_velocity`.

In [ ]:
def lean_policy(obs):
    # BEGIN SOLUTION
    angle = obs[2]
    return 1 if angle > 0 else 0
    # END SOLUTION


def better_policy(obs):
    # BEGIN SOLUTION
    angle, angular_velocity = obs[2], obs[3]
    return 1 if angle + 0.5 * angular_velocity > 0 else 0
    # END SOLUTION


lean_score = average_return(lean_policy)
better_score = average_return(better_policy)
print(f"lean_policy:   {lean_score:.1f} steps")
print(f"better_policy: {better_score:.1f} steps (500 is the maximum)")
check(lean_score > random_score, "lean_policy beats random")
check(better_score >= 195, "better_policy balances the pole (195+ counts as solving CartPole)")

Why does angular velocity help so much? Knowing *where the pole is* isn't enough: you also need to know
*where it's heading*. A pole leaning left but swinging back to the right needs a different push from
one leaning left and falling further. You'll see exactly the same thing with the balloon in notebook 2.

Notice too that you wrote these policies yourself. RL's job is to find a policy like `better_policy`
automatically, from rewards alone. That starts in notebook 3.

## Part 2 · Meet the balloon

Project Stratus flies **zero-pressure balloons**: an envelope of helium, open at the bottom, carrying
ballast that can be dropped. It has two ways to change altitude:

- **drop ballast:** the balloon gets lighter and rises
- **vent helium:** the balloon loses lift and sinks

Both are irreversible: you can't get ballast or helium back, and you only carry so much.

The project's physics model is the `Balloon` class in `src/environments/core/balloon.py`. It works out
buoyancy from the atmosphere, the gas law and drag. Here is a small helper that flies it according to
a plan. Read it through: it's the same loop as `run_episode`, but with the plan choosing the actions.

In [ ]:
def fly(plan, start_altitude=19_000.0, minutes=120):
    """Fly a balloon, carrying out `plan`, a dict of {minute: [actions]}.

    Each action is "drop" or "vent". Returns per-minute arrays of time (minutes),
    altitude (m) and vertical velocity (m/s).
    """
    balloon = Balloon(dim=1, position=np.array([start_altitude]))
    times, altitudes, velocities = [0], [balloon.altitude], [balloon.velocity]
    for minute in range(minutes):
        for action in plan.get(minute, []):
            if action == "drop":
                balloon.drop_ballast()
            elif action == "vent":
                balloon.vent_gas()
            else:
                raise ValueError(f"Unknown action {action!r}: use 'drop' or 'vent'")
        for _ in range(12):  # 12 physics steps of 5 s = 1 minute
            balloon.update(5.0)
        times.append(minute + 1)
        altitudes.append(balloon.altitude)
        velocities.append(balloon.velocity)
    return np.array(times), np.array(altitudes), np.array(velocities)


def plot_flight(times, altitudes, velocities, target=None):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
    if target is not None:
        ax1.axhspan((target - 250) / 1000, (target + 250) / 1000, color="tab:green", alpha=0.15)
    ax1.plot(times, altitudes / 1000)
    ax1.set(xlabel="time (minutes)", ylabel="altitude (km)")
    ax2.plot(times, velocities)
    ax2.axhline(0, color="grey", lw=0.5)
    ax2.set(xlabel="time (minutes)", ylabel="vertical velocity (m/s)")
    fig.tight_layout()
    plt.show()

First, do nothing for an hour. (The first flight takes a few seconds while the physics code compiles.)

In [ ]:
plot_flight(*fly({}, minutes=60))

The balloon was set up to float at 19 km, so it stays put. Now drop one lot of ballast (10 g) at the
start:

In [ ]:
plot_flight(*fly({0: ["drop"]}, minutes=60))

Three things to notice:

1. One drop doesn't move the balloon to a new altitude. It starts a **steady climb**, a little under
   0.5 m/s here, that carries on until you do something about it.
2. The climb takes a moment to get going: about a minute after one drop, and a few minutes after
   several drops at once. Actions don't take full effect straight away.
3. To stop, you have to take the lift away again. Stopping costs resources too.

Here's a drop followed by a vent 20 minutes later:

In [ ]:
plot_flight(*fly({0: ["drop"], 20: ["vent"]}, minutes=60))

The vent cancels the drop and the balloon stops, about 550 m higher. Drops and vents add up: two
drops climb faster than one, and then it takes two vents to stop.

### ✏️ Exercise 1.3 · Fly it by hand

Starting at 19,000 m, get the balloon into the band 20,000 ± 250 m and **stopped there** (vertical speed
below 0.05 m/s) by the end of a 2-hour flight. Use as few actions as you can: each one spends something
you can't get back.

Write your plan into `my_plan` as `{minute: [actions]}`. For example, `my_plan[0] = ["drop", "drop"]`
drops twice at the start.

In [ ]:
my_plan = {}
# BEGIN SOLUTION
my_plan[0] = ["drop", "drop"]
my_plan[25] = ["vent", "vent"]
# END SOLUTION

times, altitudes, velocities = fly(my_plan)
plot_flight(times, altitudes, velocities, target=20_000)
n_actions = sum(len(actions) for actions in my_plan.values())
print(f"Final altitude {altitudes[-1]:,.0f} m, velocity {velocities[-1]:+.3f} m/s, {n_actions} actions")
check(abs(altitudes[-1] - 20_000) <= 250, "finished inside the band")
check(abs(velocities[-1]) < 0.05, "stopped, rather than drifting out of the band")

🚀 **Stretch**
- What's the smallest number of actions that passes? Can you do it with two?
- Start at 21,200 m instead: `fly(my_plan, start_altitude=21_200)`.
- Try to arrive as fast as possible. Why does going faster make stopping harder?

## Summary

- RL learns a policy from rewards, by trial and error.
- Gymnasium's interface is `reset(seed)` → `(observation, info)` and `step(action)` →
  `(observation, reward, terminated, truncated, info)`.
- A good policy needs to know where things are heading, not just where they are.
- Balloon actions set a climb or sink rate. Their effects are delayed, and they spend resources you
  can't get back.

**Next:** notebook 2 wraps the balloon in a Gymnasium environment, so that a loop like `run_episode`,
and later an RL library, can fly it.

**Git checkpoint.** Save the notebook, then from the repo root:

```bash
git add onboarding/01_rl_loop_and_gymnasium.ipynb
git commit -m "Onboarding notebook 1"
git push
```

The first push of your branch needs `git push -u origin onboarding/<your-name>` instead. The pre-commit
hooks strip cell outputs when you commit, so only your code is saved.